In [ ]:
EXPERIMENT_ID = "2025-12-11_11-04-16"

PATH_TO_RESULTS_FOLDER = f".result/{EXPERIMENT_ID}/"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as constants
from wave_optics_propagation.analytics import (
    free_space_propagated_gaussian_wavefront,
    gaussian_signal,
    propagated_gaussian_wavefront_hitting_lens,
)
from wave_optics_propagation.visualization import plot_wavefunction

from wave_optics_propagation_enhanced.parameters import ExperimentParameters
from wave_optics_propagation_enhanced.result import ExperimentResult

In [ ]:
params = ExperimentParameters.from_file(path=PATH_TO_RESULTS_FOLDER)
results = ExperimentResult.from_file(path=PATH_TO_RESULTS_FOLDER)

In [ ]:
snapshots = results.snapshots


def process_statevector(statevector: np.ndarray) -> np.ndarray:
    return statevector


initial_statevector = process_statevector(snapshots[f"step_{0}"])

lens_statevectors = [
    process_statevector(snapshots[f"step_lens_{i}"])
    if f"step_lens_{i}" in snapshots
    else None
    for i in range(params.lens_slices)
]
after_lens_statevectors = [
    process_statevector(snapshots[f"step_after_lens_{i + 1}"])
    if f"step_after_lens_{i + 1}" in snapshots
    else None
    for i in range(params.num_of_steps_after_lens)
]

In [ ]:
from wave_optics_propagation_enhanced.classical_numerics import (
    classical_numerics_simulation,
    thin_lens_simulation,
)

print("Plotting initial state")
plot_wavefunction(initial_statevector, plot_size_scale=1, normalize=True)
plt.suptitle("Initial state")
plt.show()

NUM_OF_LENS_SLICES_TO_PLOT = 5
NUM_OF_AFTER_LENS_SLICES_TO_PLOT = params.num_of_steps_after_lens

for i, statevector in enumerate(lens_statevectors):
    if statevector is None:
        continue
    if (
        results.total_lenses_simulated > NUM_OF_LENS_SLICES_TO_PLOT
        and i % (results.total_lenses_simulated // NUM_OF_LENS_SLICES_TO_PLOT) != 0
    ):
        continue

    # print(f"Plotting lens slice {i + 1}/{lens_slices}")

    fig, (ax1, ax2) = plot_wavefunction(statevector, plot_size_scale=1, normalize=True)

    # z = params.lens_slice_thickness * (i + 1)
    # R_z, w_z = propagated_gaussian_wavefront_hitting_lens(
    #     params.gaussian_beam_waist,
    #     z,
    #     params.focal_length,
    #     params.vacuum_wavelength,
    #     params.refractive_index,
    # )
    # signal = gaussian_signal(params.x_axis, w_z, params.gaussian_mean)
    # ax1.plot(np.abs(signal.normalized_data), "r--", label="thin lens analytical")

    plt.suptitle(f"After lens slice {i + 1}/{params.lens_slices}")
    plt.show()

for i, statevector in enumerate(after_lens_statevectors):
    if statevector is None:
        continue
    if i % (params.num_of_steps_after_lens // NUM_OF_AFTER_LENS_SLICES_TO_PLOT) != 0:
        continue

    # print(f"Plotting free space step {i + 1}/{num_of_steps_after_lens}")
    fig, (ax1, ax2) = plot_wavefunction(statevector, plot_size_scale=1, normalize=True)

    after_lens_distance = params.step_size_after_lens * (i + 1)

    analytical_result = thin_lens_simulation(
        params, propagation_after_lens=after_lens_distance
    )

    classical_numerics_result = classical_numerics_simulation(
        params, propagation_after_lens=after_lens_distance
    )

    ax1.plot(np.abs(analytical_result), "r--", label="thin lens analytical")

    ax1.plot(np.abs(classical_numerics_result), "g--", label="classical numerics")

    plt.suptitle(f"After free space step {i + 1}/{params.num_of_steps_after_lens}")
    plt.show()